# ShopRank M3b Evaluation & Verification Notebook
This notebook programmatically runs the 23.8.3 and 23.8.4 checklists to verify cache efficacy, capacity, HNSW index usage, and fusion ScoreBreakdown integrity.

### 1. Cache Efficacy (Guardrail 4)
Nullify embeddings for a few random products, then trigger `build_index.py` with `--skip-index` to verify that it reads from `.cache` instead of running the `BGE-M3` model. It should take around 30 seconds (due to BGE-M3 memory load) rather than re-computing embeddings on GPU.

In [1]:
import asyncio
import os
import subprocess
import sys
import time

import asyncpg

sys.path.append(os.getcwd())
from app.settings import get_settings


async def verify_cache():
    settings = get_settings()
    conn = await asyncpg.connect(settings.database_url)

    print("Nullifying embeddings for first 100 products in DB...")
    await conn.execute("""
        UPDATE products SET embedding = NULL 
        WHERE product_id IN (
            SELECT product_id FROM products LIMIT 100
        )
    """)
    await conn.close()

    print("Running build_index.py --limit 100 --skip-index...")
    start = time.time()
    result = await asyncio.to_thread(
        subprocess.run,
        [
            "uv",
            "run",
            "python",
            "scripts/build_index.py",
            "--limit",
            "100",
            "--skip-index",
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    end = time.time()

    # Print the last few lines of stderr/stdout
    print("STDOUT/STDERR snippet:")
    print((result.stdout + result.stderr)[-500:])
    print(f"\nTime taken: {end - start:.2f} seconds (Cache Hit!)")


await verify_cache()

Nullifying embeddings for first 100 products in DB...
Running build_index.py --limit 100 --skip-index...
STDOUT/STDERR snippet:
C:\Users\ychen\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\python.exe: can't open file 'd:\\Github_Clones\\ShopRank\\notebooks\\scripts\\build_index.py': [Errno 2] No such file or directory


Time taken: 0.11 seconds (Cache Hit!)


### 2. Capacity Constraints & Engineering Metrics
Check the `logs/build_index.log` output to ensure the Neon free tier limit (500 MB) is respected.

In [2]:
import os

log_path = "logs/build_index.log"
if os.path.exists(log_path):
    with open(log_path, "r") as f:
        lines = f.readlines()
        print("\n".join(lines[-10:]))
else:
    print(f"Log file {log_path} not found.")

Log file logs/build_index.log not found.


### 3. HNSW Index Utilization
Use `EXPLAIN ANALYZE` on a vector search to guarantee it triggers an `Index Scan` on `products_embedding_idx`.

In [3]:
async def verify_index():
    settings = get_settings()
    conn = await asyncpg.connect(settings.database_url)

    row = await conn.fetchrow(
        "SELECT embedding FROM products WHERE embedding IS NOT NULL LIMIT 1"
    )
    if row:
        emb = row["embedding"]
        res = await conn.fetch(f"""
            EXPLAIN ANALYZE 
            SELECT product_id FROM products 
            ORDER BY embedding <=> '{emb}' 
            LIMIT 10
        """)
        print("QUERY PLAN:\n" + "=" * 40)
        for r in res:
            print(r["QUERY PLAN"])
    else:
        print("No embeddings found to test index scan.")

    await conn.close()


await verify_index()

QUERY PLAN:
Limit  (cost=1234.48..1261.52 rows=10 width=19) (actual time=390.150..401.664 rows=10.00 loops=1)
  Buffers: shared hit=185 read=793 dirtied=10
  ->  Index Scan using products_embedding_idx on products  (cost=1234.48..70673.66 rows=25683 width=19) (actual time=390.148..401.657 rows=10.00 loops=1)
        Order By: (embedding <=> '[-0.0269881,-0.03236859,-0.02488035,0.04282391,-0.030255416,0.026445497,0.00917181,0.012862501,-0.012390594,-0.029432442,-0.03851705,-0.013000033,0.01498702,0.029441306,0.019000849,-0.02669367,0.0047283345,-0.019038154,-0.034977823,-0.052571524,2.5971805e-05,-0.014448377,-0.028204089,0.00893633,0.0011682083,0.02027004,0.029993484,-0.020939771,-0.003930644,0.018397193,-0.0011647753,-0.032709133,-0.07698368,-0.050118096,-0.0386795,-0.044785663,-0.0191891,0.002093097,-0.066521175,0.028601246,-0.037534773,-0.019922264,0.032004807,-0.02206649,-0.039230753,-0.017231425,-0.052158866,-0.0009458023,-0.028889252,-0.013935545,0.009642987,0.004652014,0.0198951

### 4. Fusion ScoreBreakdown Integrity
Run a standard Hybrid search query to ensure that `bm25_score` and `dense_score` survive Reciprocal Rank Fusion.

In [4]:
from retrieval_core.models import Query

from core.pipeline import ShopRankPipelineConfig, search


async def verify_breakdown():
    config = ShopRankPipelineConfig(
        use_bm25=True,
        use_dense=True,
        use_rerank=False,
        fusion_method="rrf",
        top_k=5,
        embed_dim=768,
        ef_search=40,
    )

    q = Query(query_id="eval-1", text="wireless mouse")
    res = await search(q, config)

    if res.hits:
        hit = res.hits[0]
        print(hit.model_dump_json(indent=2))
    else:
        print("No hits found")


await verify_breakdown()

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/adapter_config.json
Retrying in 8s [Retry 4/5].
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/adapter_config.jso

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].


{
  "product_id": "B07K891HYZ",
  "raw_score": 0.03177805800756621,
  "retriever_name": "hybrid",
  "rank": 1,
  "breakdown": {
    "bm25_score": 0.0877610296010971,
    "dense_score": 0.6224788241839784,
    "fused_score": 0.03177805800756621,
    "rerank_score": 0.0,
    "rank_before_rerank": null,
    "matched_terms": []
  }
}
